<a href="https://colab.research.google.com/github/Davron030901/Machine_Learning/blob/main/yolo_realtime_face_filters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎥 Real-time YOLO Face Filters (Colab, live webcam)

Live-ish AR filters — **sunglasses, party hat, mustache** — driven by **YOLO-pose** on your webcam, streaming inside Google Colab.

**How it works**
1. **Stream** frames from your webcam through the browser into Colab.
2. **YOLO-pose** (`yolo11n-pose`) finds every person and returns keypoints — including **eyes and nose**.
3. We turn those points into filter placement (eye distance → size, eye-line tilt → rotation).
4. We draw the stickers on a **transparent overlay** and send it back to sit on top of the live video.

> ⚙️ **Turn on the GPU first:** `Runtime ▸ Change runtime type ▸ T4 GPU`. YOLO runs much faster on it.
>
> ⚠️ **Expect ~5–10 fps, a bit choppy.** In Colab every frame round-trips browser → server → browser, so the network is the bottleneck, not YOLO. For buttery 30 fps you'd run this locally (see the last section) — the detection/placement code is identical; only the capture changes.
>
> Click the video (or the red text) to **stop** the stream.


## 0. Install YOLO
Colab already has PyTorch, OpenCV, Pillow and NumPy — we just add **Ultralytics** (YOLO).

In [ ]:
!pip install -q ultralytics

In [ ]:
import io, math, time, base64
import cv2
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display, Javascript
from google.colab.output import eval_js
import torch
from ultralytics import YOLO

print("Torch CUDA available:", torch.cuda.is_available(),
      "->", "GPU 🟢" if torch.cuda.is_available() else "CPU 🟡 (works, just slower)")

# Pose model with face keypoints (nose, eyes, ears + body). Auto-downloads (~6 MB).
model = YOLO("yolo11n-pose.pt")     # fallback: YOLO("yolov8n-pose.pt")
print("YOLO-pose loaded.")

Torch CUDA available: True -> GPU 🟢
YOLO-pose loaded.


## 1. Build the filter stickers (drawn in code — nothing to download)

In [ ]:
SS = 3
def _canvas(w, h): return Image.new("RGBA", (w*SS, h*SS), (0,0,0,0))
def _finish(img, w, h): return img.resize((w, h), Image.LANCZOS)

def draw_sunglasses():
    w, h = 400, 150; img = _canvas(w, h); d = ImageDraw.Draw(img); s = SS
    black = (15,15,18,235)
    d.rounded_rectangle([25*s,35*s,180*s,120*s], radius=45*s, fill=black)
    d.rounded_rectangle([220*s,35*s,375*s,120*s], radius=45*s, fill=black)
    d.rounded_rectangle([180*s,60*s,220*s,80*s], radius=8*s, fill=black)
    d.rounded_rectangle([5*s,55*s,25*s,70*s], radius=6*s, fill=black)
    d.rounded_rectangle([375*s,55*s,395*s,70*s], radius=6*s, fill=black)
    d.ellipse([45*s,48*s,95*s,72*s], fill=(120,150,190,90))
    d.ellipse([240*s,48*s,290*s,72*s], fill=(120,150,190,90))
    return _finish(img, w, h)

def draw_party_hat():
    w, h = 240, 300; img = _canvas(w, h); d = ImageDraw.Draw(img); s = SS
    tip = (120*s, 15*s); bl = (35*s, 275*s); br = (205*s, 275*s)
    d.polygon([tip, bl, br], fill=(70,110,220,255))
    for i, col in enumerate([(255,90,120,255),(255,205,70,255),(90,220,160,255),(255,140,90,255)]):
        y = 70 + i*52; t = (y-15)/(275-15); halfw = (170*s/2)*t; cx = 120*s; yy = y*s
        d.line([(cx-halfw, yy),(cx+halfw, yy)], fill=col, width=18*s)
    d.ellipse([tip[0]-26*s, tip[1]-26*s, tip[0]+26*s, tip[1]+26*s], fill=(255,255,255,255))
    return _finish(img, w, h)

def draw_mustache():
    w, h = 300, 130; img = _canvas(w, h); d = ImageDraw.Draw(img); s = SS
    col = (35,25,20,255); cx = 150*s
    d.ellipse([cx-40*s, 30*s, cx+40*s, 95*s], fill=col)
    for sign in (-1, +1):
        for i in range(14):
            t = i/13; x = cx + sign*(40*s + int(85*s*t))
            yc = 55*s + int(18*s*math.sin(t*math.pi)); r = int((30 - 22*t)*s)
            d.ellipse([x-r, yc-r, x+r, yc+r], fill=col)
    d.polygon([(cx-18*s,25*s),(cx+18*s,25*s),(cx,55*s)], fill=(0,0,0,0))
    return _finish(img, w, h)

ASSETS = {"sunglasses": draw_sunglasses(), "hat": draw_party_hat(), "mustache": draw_mustache()}
print("Stickers ready.")

Stickers ready.


## 2. Placement geometry (same math, tunable)

In [ ]:
SUNGLASSES_SCALE = 2.35
HAT_SCALE        = 2.70
HAT_LIFT         = 2.15
MUSTACHE_SCALE   = 1.50
MUSTACHE_DROP    = 0.55

def overlay(base, acc, center, width, angle_deg):
    if width < 4: return
    ratio = acc.height / acc.width
    w = max(4, int(width)); h = max(4, int(width*ratio))
    a = acc.resize((w, h), Image.LANCZOS).rotate(angle_deg, expand=True, resample=Image.BICUBIC)
    base.alpha_composite(a, (int(center[0]-a.width/2), int(center[1]-a.height/2)))

def apply_filters(base, kp, filters):
    re, le = kp["right_eye"], kp["left_eye"]
    ex, ey = (re[0]+le[0])/2, (re[1]+le[1])/2
    dx, dy = le[0]-re[0], le[1]-re[1]
    eye_dist = math.hypot(dx, dy) or 1
    angle = math.degrees(math.atan2(dy, dx))
    ux, uy = dy, -dx; n = math.hypot(ux, uy) or 1; ux, uy = ux/n, uy/n
    if uy > 0: ux, uy = -ux, -uy
    if "sunglasses" in filters:
        overlay(base, ASSETS["sunglasses"], (ex, ey), SUNGLASSES_SCALE*eye_dist, -angle)
    if "hat" in filters:
        hc = (ex + ux*HAT_LIFT*eye_dist, ey + uy*HAT_LIFT*eye_dist)
        overlay(base, ASSETS["hat"], hc, HAT_SCALE*eye_dist, -angle)
    if "mustache" in filters:
        nose, mouth = kp["nose"], kp["mouth"]
        mc = (nose[0]+(mouth[0]-nose[0])*MUSTACHE_DROP, nose[1]+(mouth[1]-nose[1])*MUSTACHE_DROP)
        overlay(base, ASSETS["mustache"], mc, MUSTACHE_SCALE*eye_dist, -angle)
print("apply_filters() ready.")

apply_filters() ready.


## 3. YOLO keypoints → face points

COCO pose keypoints: **0 = nose, 1 = left eye, 2 = right eye**. We use those three (and synthesize a "mouth" point below the nose for the mustache). Low-confidence detections are skipped.

In [ ]:
def _faces_from_kpts(xy, conf, conf_th=0.3):
    faces = []
    for pts, c in zip(xy, conf):
        nose, leye, reye = pts[0], pts[1], pts[2]
        if c[1] < conf_th or c[2] < conf_th:        # need both eyes
            continue
        re = (float(reye[0]), float(reye[1]))
        le = (float(leye[0]), float(leye[1]))
        ex, ey = (re[0]+le[0])/2, (re[1]+le[1])/2
        dx, dy = le[0]-re[0], le[1]-re[1]
        eye_dist = math.hypot(dx, dy) or 1
        ux, uy = dy, -dx; n = math.hypot(ux, uy) or 1; ux, uy = ux/n, uy/n
        if uy > 0: ux, uy = -ux, -uy                # "up"
        dnx, dny = -ux, -uy                          # "down"
        nose_pt = (float(nose[0]), float(nose[1])) if c[0] >= conf_th \
                  else (ex + dnx*0.9*eye_dist, ey + dny*0.9*eye_dist)
        mouth_pt = (nose_pt[0] + dnx*0.9*eye_dist, nose_pt[1] + dny*0.9*eye_dist)
        faces.append({"right_eye": re, "left_eye": le, "nose": nose_pt, "mouth": mouth_pt})
    return faces

def keypoints_to_faces(result, conf_th=0.3):
    if result.keypoints is None or result.keypoints.xy is None:
        return []
    xy = result.keypoints.xy.cpu().numpy()
    cf = result.keypoints.conf
    cf = cf.cpu().numpy() if cf is not None else np.ones(xy.shape[:2])
    return _faces_from_kpts(xy, cf, conf_th)
print("keypoints_to_faces() ready.")

keypoints_to_faces() ready.


## 4. Webcam video streaming plumbing

This sets up the browser side: it opens your camera, and each loop it hands one frame to Python and draws Python's returned overlay on top of the live video. (Advanced JS — students can just run it.)

In [ ]:
def start_stream():
    js = Javascript("""
    var video, div, stream, imgElement, labelElement, captureCanvas;
    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
        if (stream) stream.getVideoTracks()[0].stop();
        if (video) video.remove();
        if (div) div.remove();
        video = null; div = null; stream = null;
        imgElement = null; captureCanvas = null; labelElement = null;
    }
    function onAnimationFrame() {
        if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
        if (pendingResolve) {
            var result = "";
            if (!shutdown && captureCanvas) {
                captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
                result = captureCanvas.toDataURL('image/jpeg', 0.8);
            }
            var lp = pendingResolve; pendingResolve = null; lp(result);
        }
    }
    async function createDom() {
        if (div && video) return stream;
        div = document.createElement('div');
        div.style.maxWidth = '640px';
        document.body.appendChild(div);

        const modelOut = document.createElement('div');
        labelElement = document.createElement('span');
        labelElement.innerText = 'Starting...';
        labelElement.style.fontWeight = 'bold';
        modelOut.appendChild(labelElement);
        div.appendChild(modelOut);

        try {
            stream = await navigator.mediaDevices.getUserMedia({video: true});
        } catch (err) {
            labelElement.innerHTML = '<span style="color:red">Error: Camera permission denied. Please allow camera access in your browser address bar and run again.</span>';
            throw err;
        }

        video = document.createElement('video');
        video.style.display = 'block';
        video.width = 640;
        video.setAttribute('playsinline', '');
        video.onclick = () => { shutdown = true; };
        div.appendChild(video);

        imgElement = document.createElement('img');
        imgElement.style.position = 'absolute';
        imgElement.style.zIndex = 1;
        imgElement.onclick = () => { shutdown = true; };
        div.appendChild(imgElement);

        const stop = document.createElement('div');
        stop.innerHTML = '<span style="color:red;font-weight:bold;">Click the video to STOP</span>';
        stop.onclick = () => { shutdown = true; };
        div.appendChild(stop);

        video.srcObject = stream;
        await video.play();
        captureCanvas = document.createElement('canvas');
        captureCanvas.width = 640; captureCanvas.height = 480;
        window.requestAnimationFrame(onAnimationFrame);
        return stream;
    }
    async function stream_frame(label, imgData) {
        if (shutdown) { removeDom(); shutdown = false; return ''; }
        try {
            await createDom();
        } catch (e) { return ''; }

        if (label != "" && labelElement) labelElement.innerHTML = label;
        if (imgData != "" && imgElement) {
            var r = video.getClientRects()[0];
            if (r) {
              imgElement.style.top = r.top + "px";
              imgElement.style.left = r.left + "px";
              imgElement.style.width = r.width + "px";
              imgElement.style.height = r.height + "px";
              imgElement.src = imgData;
            }
        }
        var result = await new Promise(function(resolve){ pendingResolve = resolve; });
        return {'img': result};
    }
    """)
    display(js)

def take_frame(label, img_data):
    return eval_js('stream_frame("{}", "{}")'.format(label, img_data))

def js_to_rgb(js_reply):
    if not js_reply or 'img' not in js_reply or not js_reply['img']:
        return None
    b = base64.b64decode(js_reply['img'].split(',')[1])
    return np.array(Image.open(io.BytesIO(b)).convert('RGB'))

def overlay_to_dataurl(rgba):
    img = Image.fromarray(rgba.astype('uint8'), 'RGBA')
    buf = io.BytesIO(); img.save(buf, format='png')
    return 'data:image/png;base64,{}'.format(base64.b64encode(buf.getvalue()).decode())

print("Streaming plumbing ready with error handling.")

Streaming plumbing ready with error handling.


## 5. ▶️ Run the live filter booth!

Allow camera access, then wave / tilt your head. **Click the video to stop.**
Change `FILTERS` to pick which stickers appear.

In [ ]:
FILTERS = ["sunglasses", "hat", "mustache"]   # e.g. ["sunglasses"] or ["hat","mustache"]

start_stream()
img_data = ""
frames = 0
t0 = time.time()
try:
    while True:
        js_reply = take_frame("YOLO filters live — click video to stop", img_data)
        if not js_reply:
            break
        frame = js_to_rgb(js_reply)                                   # RGB 640x480
        results = model(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR),
                        imgsz=480, verbose=False)
        canvas = Image.new("RGBA", (frame.shape[1], frame.shape[0]), (0,0,0,0))
        for r in results:
            for kp in keypoints_to_faces(r):
                apply_filters(canvas, kp, FILTERS)
        img_data = overlay_to_dataurl(np.array(canvas))
        frames += 1
finally:
    dt = time.time() - t0
    if frames:
        print(f"Processed {frames} frames in {dt:.1f}s  (~{frames/dt:.1f} fps)")

<IPython.core.display.Javascript object>

/tmp/ipykernel_478/3299822566.py:103: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(rgba.astype('uint8'), 'RGBA')


Processed 464 frames in 190.9s  (~2.4 fps)


## 6. Make it yours & go further

**Tinker (easy)**
- Edit `FILTERS`, or the tuning knobs in Step 2 (sizes, positions).
- Redraw / recolor the stickers in Step 1, or register your own PNG:
  `ASSETS["crown"] = Image.open("crown.png").convert("RGBA")`.

**Speed it up (medium)**
- Turn on the **T4 GPU** (`Runtime ▸ Change runtime type`).
- Run YOLO every *other* frame and reuse the last overlay in between — noticeably smoother.
- Lower `imgsz` (e.g. 320) for faster inference at some accuracy cost.

**Level up (advanced)**
- Swap `yolo11n-pose` for a dedicated **YOLO-face** model to get mouth/lip landmarks (better mustache & add lipstick).
- **Run locally for true 30 fps:** replace the streaming plumbing with
  `cap = cv2.VideoCapture(0)` and an `cv2.imshow` window — *the YOLO + placement code is unchanged.*
- **Posture tie-in:** `yolo11n-pose` already returns all 17 body keypoints (shoulders, hips…). Use the shoulder–ear angle to flash a "slouching!" warning — a direct bridge to real posture monitoring.

**Teaching takeaway:** YOLO gave us points; the filters are just **geometry + alpha compositing**. Same lesson as before, now with a detector robust to tilt, lighting, and multiple people — and you can *see* the fps cost of streaming, which is a great intro to the compute-vs-latency trade-offs in real CV systems.

---
*Troubleshooting:* black video / no prompt → click **Allow** for the camera and use Chrome. Very slow → enable GPU, or lower `imgsz`. Weights won't download → re-run the load cell (needs internet to fetch `yolo11n-pose.pt` once).
